In [1]:
import torch

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
print(f"MPS (Apple GPU) disponible : {torch.backends.mps.is_available()}")

if torch.cuda.is_available():
    print(f"GPU CUDA : {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    print("GPU Apple Silicon (MPS) détecté")
    device = torch.device("mps")
else:
    print("Aucun GPU détecté, utilisation du CPU")
    device = torch.device("cpu")

print(f"\nDevice utilisé : {device}")

# Test rapide sur le device
x = torch.randn(3, 3).to(device)
print(f"Tenseur de test sur {device} :\n{x}")


PyTorch version : 2.9.0+cu126
CUDA disponible : True
MPS (Apple GPU) disponible : False
GPU CUDA : Tesla T4

Device utilisé : cuda
Tenseur de test sur cuda :
tensor([[-0.1778,  0.1138,  0.0571],
        [ 0.8506, -1.0359, -1.4355],
        [ 0.2934, -0.4116,  0.0336]], device='cuda:0')


# Entraînement SO-ARM100 — Reach & Pick-and-Place

Pipeline SAC (Stable-Baselines3) en deux phases :

1. **Reach** : l'effecteur (milieu de la pince) atteint le cube vert (~500k steps)
2. **Pick-and-place** : approche **pince ouverte centrée** sur le cube → fermeture → lever → transport → dépôt (~2M steps)

### Comportement de saisie ciblé
`AlignedPickPlaceEnv` ajoute trois termes de récompense pour que le cube soit bien centré entre les mors avant la fermeture :
- **Récompense pince ouverte** pendant l'approche finale
- **Bonus de centrage** : pince ouverte + cube entre les mors
- **Malus de fermeture prématurée** si la pince se ferme avant le centrage

> Pré-requis : `mujoco`, `gymnasium`, `stable-baselines3[extra]` installés dans le kernel actif.

In [14]:
%%writefile reach_cube_env.py
"""
reach_cube_env.py
-----------------
Gymnasium environment for the SO-ARM100 to reach the green cube in TEST_SCENE.xml.
"""
from __future__ import annotations

import math
import pathlib
from typing import Any

import gymnasium as gym
import mujoco
import numpy as np
from gymnasium.spaces import Box

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
_THIS_DIR = pathlib.Path(__file__).parent
SCENE_PATH = _THIS_DIR / "trs_so_arm100" / "TEST_SCENE.xml"

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
HOME_QPOS = np.array([0.0, -1.57, 1.57, 1.57, -1.57, 0.0], dtype=np.float64)
JOINT_DELTA_SCALE = np.array([0.08, 0.05, 0.05, 0.04, 0.04, 0.02], dtype=np.float64)
CUBE_REST_Z = 0.22
REACH_THRESHOLD = 0.05
CUBE_X_RANGE = (0.26, 0.54)
CUBE_Y_RANGE = (-0.13, 0.0)
ACTUATOR_NAMES = ["Rotation", "Pitch", "Elbow", "Wrist_Pitch", "Wrist_Roll", "Jaw"]
SAFE_PITCH_LOW  = -1.95
SAFE_ELBOW_LOW  =  0.05

GOAL_POSITIONS = np.array([
    [ 0.0,  0.4, 0.17],
    [-0.4,  0.0, 0.17],
    [ 0.0, -0.4, 0.17],
], dtype=np.float32)

GRASP_REACH        = 0.05
GRASP_JAW_THRESH   = 0.20
RELEASE_JAW_THRESH = 0.55
LIFT_HEIGHT        = 0.05
PLACE_THRESHOLD    = 0.08


class ReachCubeEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 50}

    def __init__(
        self,
        scene_path=None,
        use_yolo: bool = False,
        task: str = "reach",
        render_mode=None,
        max_episode_steps: int = 500,
        frame_skip: int = 8,
        random_cube: bool = True,
        seed=None,
        cube_x_range=None,
        cube_y_range=None,
    ) -> None:
        super().__init__()
        assert task in ("reach", "pick_place"), f"Unknown task '{task}'"
        assert render_mode in (None, "human", "rgb_array"), f"Unknown render_mode '{render_mode}'"

        self.scene_path     = pathlib.Path(scene_path) if scene_path else SCENE_PATH
        self.use_yolo       = use_yolo
        self.task           = task
        self.render_mode    = render_mode
        self.max_episode_steps = max_episode_steps
        self.frame_skip     = frame_skip
        self.random_cube    = random_cube
        self.cube_x_range   = cube_x_range if cube_x_range is not None else CUBE_X_RANGE
        self.cube_y_range   = cube_y_range if cube_y_range is not None else CUBE_Y_RANGE

        self.model = mujoco.MjModel.from_xml_path(str(self.scene_path))
        self.data  = mujoco.MjData(self.model)
        self.np_random = np.random.default_rng(seed)

        self._resolve_ids()

        self.ctrl_low  = self.model.actuator_ctrlrange[:, 0].copy()
        self.ctrl_high = self.model.actuator_ctrlrange[:, 1].copy()
        pitch_idx  = self._act_ids.index(self._act_id("Pitch"))
        elbow_idx  = self._act_ids.index(self._act_id("Elbow"))
        self.ctrl_low[pitch_idx] = max(self.ctrl_low[pitch_idx], SAFE_PITCH_LOW)
        self.ctrl_low[elbow_idx] = max(self.ctrl_low[elbow_idx], SAFE_ELBOW_LOW)

        obs_dim = 21 if self.task == "reach" else 29
        self.action_space      = Box(-1.0, 1.0, shape=(6,), dtype=np.float32)
        self.observation_space = Box(-np.inf, np.inf, shape=(obs_dim,), dtype=np.float32)

        self._episode_step    = 0
        self._reached_once    = False
        self._grasped_once    = False
        self._placed_once     = False
        self._attached        = False
        self._attach_offset   = np.zeros(3, dtype=np.float64)
        self._goal_pos        = GOAL_POSITIONS[0].copy()
        self._cube_obs_pos    = np.zeros(3, dtype=np.float32)
        self.last_reward      = 0.0
        self.reward_info: dict = {}

        self._renderer = None
        self._viewer   = None
        if render_mode == "rgb_array":
            self._renderer = mujoco.Renderer(self.model, 480, 640)
        elif render_mode == "human":
            import mujoco.viewer as mv
            self._viewer = mv.launch_passive(self.model, self.data)

        self._yolo_detector = None

    def _act_id(self, name: str) -> int:
        i = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_ACTUATOR, name)
        if i < 0:
            raise RuntimeError(f"Actuator '{name}' not found.")
        return i

    def _resolve_ids(self) -> None:
        def _body(name):
            i = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_BODY, name)
            if i < 0: raise RuntimeError(f"Body '{name}' not found.")
            return i
        def _jnt(name):
            i = mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_JOINT, name)
            if i < 0: raise RuntimeError(f"Joint '{name}' not found.")
            return i

        self._obj_body_id   = _body("object")
        self._fixed_jaw_id  = _body("Fixed_Jaw")
        self._moving_jaw_id = _body("Moving_Jaw")

        obj_jnt = _jnt("object_joint")
        self._obj_qpos_adr = int(self.model.jnt_qposadr[obj_jnt])
        self._obj_qvel_adr = int(self.model.jnt_dofadr[obj_jnt])

        rot_jnt = _jnt("Rotation")
        self._arm_qpos_start = int(self.model.jnt_qposadr[rot_jnt])
        self._arm_qvel_start = int(self.model.jnt_dofadr[rot_jnt])

        self._act_ids = [self._act_id(n) for n in ACTUATOR_NAMES]

    def _ee_pos(self) -> np.ndarray:
        fixed  = self.data.xpos[self._fixed_jaw_id]
        moving = self.data.xpos[self._moving_jaw_id]
        return ((fixed + moving) * 0.5).astype(np.float32)

    def _cube_pos_gt(self) -> np.ndarray:
        return self.data.xpos[self._obj_body_id].astype(np.float32)

    def _get_obs(self) -> np.ndarray:
        qpos = self.data.qpos[self._arm_qpos_start : self._arm_qpos_start + 6].astype(np.float32)
        qvel = self.data.qvel[self._arm_qvel_start : self._arm_qvel_start + 6].astype(np.float32)
        ee   = self._ee_pos()
        cube = self._cube_obs_pos
        vec  = cube - ee
        base = np.concatenate([qpos, qvel, ee, cube, vec], dtype=np.float32)
        if self.task == "reach":
            return base
        goal         = self._goal_pos.astype(np.float32)
        cube_to_goal = goal - cube
        jaw_open     = np.array([self._jaw_open_frac()], dtype=np.float32)
        grasped      = np.array([1.0 if self._attached else 0.0], dtype=np.float32)
        return np.concatenate([base, goal, cube_to_goal, jaw_open, grasped], dtype=np.float32)

    def _jaw_open_frac(self) -> float:
        jaw_act_id = self._act_ids[5]
        ctrl = float(self.data.ctrl[jaw_act_id])
        lo   = float(self.ctrl_low[jaw_act_id])
        hi   = float(self.ctrl_high[jaw_act_id])
        return float(np.clip((ctrl - lo) / (hi - lo + 1e-9), 0.0, 1.0))

    def _apply_assist_grasp(self) -> None:
        jaw_frac = self._jaw_open_frac()
        if not self._attached:
            ee   = self._ee_pos().astype(np.float64)
            cube = self._cube_pos_gt().astype(np.float64)
            dist = float(np.linalg.norm(ee - cube))
            if dist < GRASP_REACH and jaw_frac < GRASP_JAW_THRESH:
                self._attached      = True
                self._grasped_once  = True
                self._attach_offset = cube - ee
        if self._attached:
            if jaw_frac > RELEASE_JAW_THRESH:
                self._attached = False
            else:
                ee      = self._ee_pos().astype(np.float64)
                new_pos = ee + self._attach_offset
                self.data.qpos[self._obj_qpos_adr     : self._obj_qpos_adr + 3] = new_pos
                self.data.qpos[self._obj_qpos_adr + 3 : self._obj_qpos_adr + 7] = [1.0, 0.0, 0.0, 0.0]
                self.data.qvel[self._obj_qvel_adr      : self._obj_qvel_adr + 6] = 0.0

    def _compute_pick_place_reward(self) -> float:
        r = 0.0
        ee       = self._ee_pos()
        cube_gt  = self._cube_pos_gt()
        goal     = self._goal_pos.astype(np.float32)

        dist_ee_cube   = float(np.linalg.norm(ee   - cube_gt))
        dist_cube_goal = float(np.linalg.norm(cube_gt - goal))
        cube_z         = float(cube_gt[2])

        self.reward_info = {
            "dist_ee_cube":   dist_ee_cube,
            "dist_cube_goal": dist_cube_goal,
            "attached":       float(self._attached),
        }

        if not self._attached:
            r += -dist_ee_cube
        else:
            if self._grasped_once:
                r += 5.0
                self._grasped_once = False
            lift = max(0.0, cube_z - CUBE_REST_Z)
            r += lift * 5.0
            r += -dist_cube_goal * 2.0

        if not self._attached and dist_cube_goal < PLACE_THRESHOLD:
            r += 10.0
            if not self._placed_once:
                r += 30.0
                self._placed_once = True
                self.reward_info["placed"] = 1.0

        r += self._joint_limit_penalty()
        return r

    def _place_cube(self) -> None:
        if self.random_cube:
            x = float(self.np_random.uniform(*self.cube_x_range))
            y = float(self.np_random.uniform(*self.cube_y_range))
        else:
            x = float((self.cube_x_range[0] + self.cube_x_range[1]) / 2)
            y = 0.0
        z = CUBE_REST_Z
        self.data.qpos[self._obj_qpos_adr     : self._obj_qpos_adr + 3] = [x, y, z]
        self.data.qpos[self._obj_qpos_adr + 3 : self._obj_qpos_adr + 7] = [1.0, 0.0, 0.0, 0.0]
        self.data.qvel[self._obj_qvel_adr      : self._obj_qvel_adr + 6] = 0.0

    def _get_cube_detection(self) -> np.ndarray:
        if not self.use_yolo:
            return self._cube_pos_gt()
        if self._yolo_detector is None:
            from yolo_detector import CubeDetector
            self._yolo_detector = CubeDetector(self.model, self.data)
        detected = self._yolo_detector.detect(self.data, cube_world_z=CUBE_REST_Z)
        if detected is None:
            return self._cube_pos_gt()
        return detected

    def _compute_reward(self, ee: np.ndarray, cube: np.ndarray) -> float:
        if self.task == "pick_place":
            return self._compute_pick_place_reward()
        dist = float(np.linalg.norm(ee - cube))
        r = -dist
        self.reward_info = {"dist": dist}
        if dist < REACH_THRESHOLD:
            r += 2.0
            self.reward_info["close_bonus"] = 2.0
            if not self._reached_once:
                r += 10.0
                self._reached_once = True
                self.reward_info["reach_bonus"] = 10.0
        joint_penalty = self._joint_limit_penalty()
        r += joint_penalty
        if joint_penalty < 0.0:
            self.reward_info["joint_penalty"] = joint_penalty
        return r

    def _joint_limit_penalty(self) -> float:
        penalty = 0.0
        qpos = self.data.qpos[self._arm_qpos_start : self._arm_qpos_start + 6]
        for i, act_id in enumerate(self._act_ids):
            lo, hi = self.ctrl_low[act_id], self.ctrl_high[act_id]
            margin = 0.05 * (hi - lo)
            if qpos[i] < lo + margin:
                penalty -= (lo + margin - qpos[i]) * 10.0
            elif qpos[i] > hi - margin:
                penalty -= (qpos[i] - (hi - margin)) * 10.0
        return penalty

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.np_random = np.random.default_rng(seed)
        mujoco.mj_resetData(self.model, self.data)
        self.data.qpos[self._arm_qpos_start : self._arm_qpos_start + 6] = HOME_QPOS
        for i, act_id in enumerate(self._act_ids):
            self.data.ctrl[act_id] = HOME_QPOS[i]
        self._place_cube()
        mujoco.mj_forward(self.model, self.data)
        self._cube_obs_pos = self._get_cube_detection()
        self._episode_step  = 0
        self._reached_once  = False
        self._grasped_once  = False
        self._placed_once   = False
        self._attached      = False
        self._attach_offset = np.zeros(3, dtype=np.float64)
        if self.task == "pick_place":
            self._goal_pos = GOAL_POSITIONS[
                int(self.np_random.integers(0, len(GOAL_POSITIONS)))
            ].copy()
        self.last_reward  = 0.0
        self.reward_info  = {}
        if self._viewer is not None:
            self._viewer.sync()
        return self._get_obs(), {}

    def step(self, action: np.ndarray):
        action = np.clip(action, -1.0, 1.0).astype(np.float64)
        for i, act_id in enumerate(self._act_ids):
            current  = float(self.data.ctrl[act_id])
            delta    = float(action[i]) * JOINT_DELTA_SCALE[i]
            new_ctrl = float(np.clip(current + delta, self.ctrl_low[act_id], self.ctrl_high[act_id]))
            self.data.ctrl[act_id] = new_ctrl
        mujoco.mj_step(self.model, self.data, nstep=self.frame_skip)
        if self.task == "pick_place":
            self._apply_assist_grasp()
            self._cube_obs_pos = self._cube_pos_gt()
        ee     = self._ee_pos()
        cube   = self._cube_obs_pos
        reward = self._compute_reward(ee, cube)
        self.last_reward = reward
        self._episode_step += 1
        terminated = False
        truncated  = self._episode_step >= self.max_episode_steps
        cube_gt = self._cube_pos_gt()
        if cube_gt[2] < 0.05 and not self._attached:
            terminated = True
            reward -= 5.0
        if self.task == "pick_place" and self._placed_once:
            terminated = True
        if self._viewer is not None:
            self._viewer.sync()
        dist_key = "dist_cube_goal" if self.task == "pick_place" else "distance"
        success  = self._placed_once if self.task == "pick_place" else self._reached_once
        info = {
            "distance": float(self.reward_info.get(dist_key, np.linalg.norm(ee - cube))),
            "success":  success,
            **self.reward_info,
        }
        return self._get_obs(), float(reward), terminated, truncated, info

    def render(self):
        if self.render_mode == "rgb_array" and self._renderer is not None:
            self._renderer.update_scene(self.data)
            return self._renderer.render()
        return None

    def close(self) -> None:
        if self._renderer is not None:
            self._renderer.close()
        if self._viewer is not None:
            self._viewer.close()
        if self._yolo_detector is not None:
            self._yolo_detector.close()

Overwriting reach_cube_env.py


In [20]:
import base64, zipfile, io, pathlib

SCENE_DST = pathlib.Path("trs_so_arm100")
if SCENE_DST.exists():
    print("✓ trs_so_arm100/ déjà présent")
else:
    print("Extraction depuis les données embarquées...")
    _zip_bytes = base64.b64decode(
        "..."
    )
    with zipfile.ZipFile(io.BytesIO(_zip_bytes)) as _z:
        _z.extractall(".")
    print("✓ trs_so_arm100/ extrait.")

SCENE_PATH = str(pathlib.Path("trs_so_arm100") / "TEST_SCENE.xml")
assert pathlib.Path(SCENE_PATH).exists(), f"TEST_SCENE.xml introuvable → {SCENE_PATH}"
print(f"✓ SCENE_PATH = {SCENE_PATH}")


Extraction depuis les données embarquées...
✓ trs_so_arm100/ extrait.
✓ SCENE_PATH = trs_so_arm100/TEST_SCENE.xml


In [22]:
import sys
import pathlib

# reach_cube_env.py a été écrit dans le répertoire courant par la cellule précédente
import numpy as np
import torch
import matplotlib.pyplot as plt

try:
    from stable_baselines3 import SAC
    from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
    from stable_baselines3.common.monitor import Monitor
    from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
    print("stable-baselines3 OK")
except ImportError as e:
    raise ImportError(
        f"stable-baselines3 introuvable : {e}\n"
        "Installez avec :  !pip install stable-baselines3[extra]"
    )

from reach_cube_env import (
    ReachCubeEnv,
    GRASP_REACH,
    GRASP_JAW_THRESH,
    CUBE_REST_Z,
    REACH_THRESHOLD,
)
print("reach_cube_env OK")

# SIM_DIR = répertoire courant (pour les outputs)
SIM_DIR = pathlib.Path(".")

# device défini en cellule 1
try:
    print(f"\nDevice actif : {device}")
except NameError:
    device = torch.device("cpu")
    print(f"device non défini → CPU : {device}")

stable-baselines3 OK
reach_cube_env OK

Device actif : cuda


In [23]:
class AlignedPickPlaceEnv(ReachCubeEnv):
    """
    Extension de ReachCubeEnv qui ajoute une récompense d'alignement pré-saisie.

    Objectif : le robot doit approcher avec la pince OUVERTE et CENTRÉE sur le cube
    avant de fermer les mors. Trois termes supplémentaires (actifs seulement en
    phase pick_place, avant la saisie) :

      1. +jaw_frac * JAW_OPEN_SCALE  — récompense la pince ouverte quand < ALIGN_DIST du cube
      2. +CENTER_BONUS               — bonus quand cube dans la pince ouverte (< CENTER_DIST ET jaw > CENTER_JAW_MIN)
      3. -PREMATURE_CLOSE_PENALTY    — malus si la pince se ferme avant d'être centrée
    """

    # Paramètres de récompense d'alignement
    ALIGN_DIST             = 0.12   # (m) distance en-dessous de laquelle la récompense s'active
    CENTER_DIST            = GRASP_REACH  # = 0.05 m  — cube "centré" dans la pince
    CENTER_JAW_MIN         = 0.55   # fraction de pince ouverte requise pour le bonus de centrage
    JAW_OPEN_SCALE         = 0.8    # poids du terme "pince ouverte"
    CENTER_BONUS           = 3.0    # bonus de centrage parfait
    PREMATURE_CLOSE_PENALTY= 2.0    # malus fermeture prématurée

    def _compute_pick_place_reward(self) -> float:
        # Récompense de base depuis la classe parente (approche → saisie → transport → dépôt)
        r = super()._compute_pick_place_reward()

        if not self._attached:
            ee      = self._ee_pos()
            cube_gt = self._cube_pos_gt()
            dist    = float(np.linalg.norm(ee - cube_gt))
            jaw_frac = self._jaw_open_frac()

            if dist < self.ALIGN_DIST:
                # 1. Pince ouverte pendant l'approche finale
                r += jaw_frac * self.JAW_OPEN_SCALE

                # 2. Bonus : cube bien centré entre les mors, pince encore ouverte
                if dist < self.CENTER_DIST and jaw_frac > self.CENTER_JAW_MIN:
                    r += self.CENTER_BONUS

                # 3. Malus : fermeture avant centrage (pince fermée mais cube pas encore centré)
                if dist > self.CENTER_DIST and jaw_frac < GRASP_JAW_THRESH:
                    r -= self.PREMATURE_CLOSE_PENALTY

        return r

In [32]:
# ─────────────────────────────────────────────────────────────────────────────
# Configuration des deux phases
# ─────────────────────────────────────────────────────────────────────────────

REACH_CFG = dict(
    task          = "reach",
    env_class     = ReachCubeEnv,
    timesteps     = 500_000,
    max_ep_steps  = 500,
    frame_skip    = 8,
    lr            = 3e-4,
    batch_size    = 256,
    buffer_size   = 500_000,
    eval_freq     = 20_000,
    eval_episodes = 10,
    seed          = 0,
    output_dir    = SIM_DIR / "outputs" / "reach_sac_nb",
)

PICK_PLACE_CFG = dict(
    task          = "pick_place",
    env_class     = AlignedPickPlaceEnv,
    timesteps     = 2_000_000,
    max_ep_steps  = 600,
    frame_skip    = 8,
    lr            = 3e-4,
    batch_size    = 256,
    buffer_size   = 1_000_000,
    eval_freq     = 40_000,
    eval_episodes = 10,
    seed          = 0,
    output_dir    = SIM_DIR / "outputs" / "pick_place_sac_nb",
    # load_model  = str(SIM_DIR / "outputs" / "reach_sac_nb" / "best_model.zip"),
)

print("Configuration REACH :")
for k, v in REACH_CFG.items():
    print(f"  {k:15s} = {v}")
print("\nConfiguration PICK-AND-PLACE :")
for k, v in PICK_PLACE_CFG.items():
    print(f"  {k:15s} = {v}")

Configuration REACH :
  task            = reach
  env_class       = <class 'reach_cube_env.ReachCubeEnv'>
  timesteps       = 500000
  max_ep_steps    = 500
  frame_skip      = 8
  lr              = 0.0003
  batch_size      = 256
  buffer_size     = 500000
  eval_freq       = 20000
  eval_episodes   = 10
  seed            = 0
  output_dir      = outputs/reach_sac_nb

Configuration PICK-AND-PLACE :
  task            = pick_place
  env_class       = <class '__main__.AlignedPickPlaceEnv'>
  timesteps       = 2000000
  max_ep_steps    = 600
  frame_skip      = 8
  lr              = 0.0003
  batch_size      = 256
  buffer_size     = 1000000
  eval_freq       = 40000
  eval_episodes   = 10
  seed            = 0
  output_dir      = outputs/pick_place_sac_nb


In [33]:
def train_sac(cfg: dict) -> SAC:
    EnvClass   = cfg["env_class"]
    task       = cfg["task"]
    scene_path = cfg.get("scene_path", SCENE_PATH)
    output_dir = pathlib.Path(cfg["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    log_dir = output_dir / "logs"
    log_dir.mkdir(exist_ok=True)

    def _make_env(rank: int = 0):
        def _init():
            env = EnvClass(
                scene_path        = scene_path,
                task              = task,
                render_mode       = None,
                max_episode_steps = cfg["max_ep_steps"],
                frame_skip        = cfg["frame_skip"],
                random_cube       = True,
                seed              = cfg["seed"] + rank,
            )
            return Monitor(env, str(output_dir))
        return _init

    train_env = VecNormalize(
        DummyVecEnv([_make_env(0)]),
        norm_obs=True, norm_reward=True, clip_obs=10.0, gamma=0.99,
    )
    eval_env = VecNormalize(
        DummyVecEnv([_make_env(99)]),
        norm_obs=True, norm_reward=False, clip_obs=10.0, training=False,
    )

    eval_cb = EvalCallback(
        eval_env,
        best_model_save_path = str(output_dir),
        log_path             = str(log_dir),
        eval_freq            = cfg["eval_freq"],
        n_eval_episodes      = cfg["eval_episodes"],
        deterministic        = True,
        verbose              = 1,
    )
    ckpt_cb = CheckpointCallback(
        save_freq   = cfg["eval_freq"],
        save_path   = str(output_dir / "checkpoints"),
        name_prefix = f"sac_{task}",
        verbose     = 1,
    )

    load_model = cfg.get("load_model", "")
    if load_model:
        print(f"Reprise depuis : {load_model}")
        model = SAC.load(load_model, env=train_env, device="auto",
                         learning_rate=cfg["lr"], batch_size=cfg["batch_size"])
    else:
        model = SAC(
            policy          = "MlpPolicy",
            env             = train_env,
            learning_rate   = cfg["lr"],
            buffer_size     = cfg["buffer_size"],
            batch_size      = cfg["batch_size"],
            learning_starts = 5_000,
            train_freq      = 1,
            gradient_steps  = 1,
            ent_coef        = "auto",
            gamma           = 0.99,
            tau             = 0.005,
            policy_kwargs   = dict(net_arch=[256, 256]),
            tensorboard_log = str(log_dir),
            verbose         = 1,
            seed            = cfg["seed"],
            device          = "auto",
        )

    print(f"\n{'='*60}")
    print(f"  Tâche      : {task.upper()}")
    print(f"  Env        : {EnvClass.__name__}")
    print(f"  Steps      : {cfg['timesteps']:,}")
    print(f"  Scene      : {scene_path}")
    print(f"  Output     : {output_dir}")
    print(f"  Device     : {model.device}")
    print(f"{'='*60}\n")

    model.learn(
        total_timesteps     = cfg["timesteps"],
        callback            = [eval_cb, ckpt_cb],
        reset_num_timesteps = not bool(load_model),
        progress_bar        = True,
    )

    model.save(str(output_dir / "final_model"))
    train_env.save(str(output_dir / "vecnorm.pkl"))
    print(f"\n✓ Entraînement terminé → {output_dir}")
    train_env.close()
    eval_env.close()
    return model

print("Fonction train_sac() définie.")

Fonction train_sac() définie.


## Phase 1 — Reach (~500k steps)

Le robot apprend à amener le **centre de la pince** sur le cube.  
Récompense : `-dist(EE, cube)` + bonus de proximité.  
Durée estimée : **30–60 min** sur CPU.

In [ ]:
import pathlib, shutil

OUTPUT_DIR = pathlib.Path(REACH_CFG["output_dir"]).resolve()

print(f"=== Fichiers reach sauvegardés dans : {OUTPUT_DIR} ===")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        print(f"  {str(f.relative_to(OUTPUT_DIR)):50s}  {f.stat().st_size:>10,} o")

# ── Option A : Télécharger sur votre machine (Colab) ─────────────────────────
# Décommentez les lignes ci-dessous pour télécharger les modèles.
# from google.colab import files as _cf
# for _f in ["best_model.zip", "final_model.zip", "vecnorm.pkl"]:
#     _p = OUTPUT_DIR / _f
#     if _p.exists():
#         _cf.download(str(_p))
#         print(f"✓ Téléchargement lancé : {_f}")

# ── Option B : Sauvegarder sur Google Drive (Colab) ──────────────────────────
# Décommentez pour copier tout le dossier sur Drive.
from google.colab import drive as _drive
_drive.mount("/content/drive")
_dest = pathlib.Path("/content/drive/MyDrive/so100_reach_sac")
shutil.copytree(str(OUTPUT_DIR), str(_dest), dirs_exist_ok=True)
print(f"✓ Copié sur Google Drive : {_dest}")

Mounted at /content/drive
✓ Copié sur Google Drive : /content/drive/MyDrive/so100_reach_sac


## Phase 2 — Pick-and-Place (~2M steps)

Le robot apprend à :
1. **Approcher** le cube **pince ouverte** (mors écartés autour du cube)
2. **Centrer** le cube entre les mors avant de fermer
3. **Lever** le cube, le **transporter** vers une plateforme colorée
4. **Déposer** le cube sur la plateforme cible

`AlignedPickPlaceEnv` guide la saisie via des récompenses d'alignement.  
Durée estimée : **3–6 h** sur CPU (recommandé : réduire `timesteps` pour un premier test).

In [ ]:
# ── Phase 2 : Pick-and-Place ──────────────────────────────────────────────────
# Optionnel : décommenter load_model dans PICK_PLACE_CFG pour partir du reach pré-entraîné.
# Lancez cette cellule pour démarrer l'entraînement "pick_place".

pick_place_model = train_sac(PICK_PLACE_CFG)

## Visualisation des courbes d'entraînement

Affiche les récompenses moyennes par épisode issues des fichiers `monitor.csv` générés pendant l'entraînement.

In [ ]:
def plot_training_curves(*cfgs, window: int = 50):
    """
    Trace les courbes de récompense lissées pour chaque configuration donnée.
    Lit les fichiers monitor.csv dans cfg['output_dir'].
    """
    import pandas as pd

    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

    for cfg, color in zip(cfgs, colors):
        output_dir = pathlib.Path(cfg["output_dir"])
        monitor_files = list(output_dir.glob("*.monitor.csv"))
        if not monitor_files:
            print(f"[{cfg['task']}] Aucun fichier monitor.csv trouvé dans {output_dir}")
            continue

        dfs = []
        for f in monitor_files:
            df = pd.read_csv(f, skiprows=1)
            dfs.append(df)
        data = pd.concat(dfs).sort_values("t").reset_index(drop=True)

        rewards = data["r"].astype(float)
        smoothed = rewards.rolling(window, min_periods=1).mean()
        episodes = np.arange(len(smoothed))

        ax.plot(episodes, smoothed, color=color, label=f"{cfg['task']} (lissé ×{window})", linewidth=2)
        ax.fill_between(
            episodes,
            rewards.rolling(window, min_periods=1).min(),
            rewards.rolling(window, min_periods=1).max(),
            alpha=0.15, color=color,
        )

    ax.set_xlabel("Épisode")
    ax.set_ylabel("Récompense par épisode")
    ax.set_title("Courbes d'entraînement SAC — SO-ARM100")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Tracez les courbes après entraînement :
plot_training_curves(REACH_CFG, PICK_PLACE_CFG)